# Lusaber · Լուսաբեր — Phase 3 training

Fine-tunes **XLM-RoBERTa** for binary Armenian-language credibility
classification on `data/armenian_news_labeled.csv` (~2 300 rows), then
trains the **LightGBM** companion on TF-IDF + Phase-2 features, and
stitches them together with the **soft-voting + calibration** stack
described in Phase 3c.

**Runtime requirements** — Google Colab with a T4 GPU (or better).
CPU runs will not complete in any reasonable time. Use
`Runtime → Change runtime type → T4 GPU` before running.

**Repo layout assumption.** The cell below expects this notebook to
live in `notebooks/` of the Lusaber repo so that `from models.features
import FeatureExtractor` resolves. If you uploaded the notebook to
Drive instead, set `REPO_PATH` accordingly.

In [ ]:
# === 0. Repo path + deps ===
import os, sys, subprocess

REPO_PATH = os.environ.get("LUSABER_REPO", "/content/lusaber")
if not os.path.isdir(REPO_PATH):
    raise SystemExit(
        f"Lusaber repo not found at {REPO_PATH}. "
        "Either `!git clone <your-fork> /content/lusaber` or set the\n"
        "LUSABER_REPO environment variable to your repo path."
    )
os.chdir(REPO_PATH)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Install only what Colab doesn't already provide. Pinned to match
# requirements.txt as closely as Colab tolerates.
PKGS = [
    "transformers==4.40.0", "datasets==2.19.0", "evaluate==0.4.1",
    "accelerate==0.29.3", "sentencepiece==0.2.0",
    "lightgbm==4.3.0", "scikit-learn==1.4.2",
    "spacy==3.7.4", "python-Levenshtein==0.25.1", "PyYAML",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PKGS])
# spaCy multilingual NER
subprocess.check_call([sys.executable, "-m", "spacy", "download", "-q", "xx_ent_wiki_sm"])
print("deps OK; cwd =", os.getcwd())

In [ ]:
# === 1. Imports + seed ===
from __future__ import annotations
import json, logging, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s :: %(message)s")
log = logging.getLogger("lusaber.train")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
log.info("device=%s  torch=%s", DEVICE, torch.__version__)
if DEVICE == "cpu":
    log.warning("No GPU detected — training on CPU will be impractical.")

In [ ]:
# === 2. Load dataset ===
CSV_PATH = Path("data/armenian_news_labeled.csv")
assert CSV_PATH.exists(), f"{CSV_PATH} not found — did the repo include the dataset?"
df = pd.read_csv(CSV_PATH)
df["label"] = df["label"].astype(int)
df["title"] = df["title"].fillna("")
df["body_text"] = df["body_text"].fillna("")
df["url"] = df["url"].fillna("")
log.info("rows=%d   label dist=%s", len(df), df["label"].value_counts().to_dict())
df.head(3)

In [ ]:
# === 3. Stratified 80/10/10 split (train / val / test) ===
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    cls = part["label"].value_counts().to_dict()
    log.info("%-5s  n=%4d   class0=%-4d class1=%-4d", name, len(part), cls.get(0,0), cls.get(1,0))

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
# === 4. Input formatter + tokenizer ===
from transformers import AutoTokenizer

BASE_MODEL = "xlm-roberta-base"
MAX_LEN = 512

def format_input(title: str, body: str) -> str:
    """Spec format: ``title + ' [SEP] ' + body_text[:1000]``."""
    title = (title or "").strip()
    body = (body or "").strip()[:1000]
    return f"{title} [SEP] {body}" if title else body

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
log.info("tokenizer loaded: vocab=%d, model_max_len=%d", tokenizer.vocab_size, tokenizer.model_max_length)

# Smoke-test on one row.
sample = train_df.iloc[0]
sample_text = format_input(sample["title"], sample["body_text"])
enc = tokenizer(sample_text, max_length=MAX_LEN, truncation=True, return_tensors="pt")
print("sample text   :", sample_text[:160], "…" if len(sample_text) > 160 else "")
print("input_ids[:25]:", enc["input_ids"][0][:25].tolist())
print("decoded[:25]  :", tokenizer.decode(enc["input_ids"][0][:25]))
print("length        :", enc["input_ids"].shape[1])

In [ ]:
# === 5. HF Datasets for the Trainer ===
from datasets import Dataset

def to_hf(part_df):
    return Dataset.from_dict({
        "text": [format_input(t, b) for t, b in zip(part_df["title"], part_df["body_text"])],
        "label": part_df["label"].tolist(),
    })

ds_train, ds_val, ds_test = to_hf(train_df), to_hf(val_df), to_hf(test_df)

def tok_batch(batch):
    return tokenizer(batch["text"], max_length=MAX_LEN, truncation=True, padding=False)

ds_train = ds_train.map(tok_batch, batched=True, remove_columns=["text"])
ds_val = ds_val.map(tok_batch, batched=True, remove_columns=["text"])
ds_test = ds_test.map(tok_batch, batched=True, remove_columns=["text"])

In [ ]:
# === 6. Model + compute_metrics ===
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

id2label = {0: "disinformation", 1: "credible"}
label2id = {v: k for k, v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2, id2label=id2label, label2id=label2id
)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    return {
        "accuracy": float(accuracy_score(labels, preds)),
        "f1": float(f1_score(labels, preds, average="macro")),
        "precision": float(precision_score(labels, preds, average="macro", zero_division=0)),
        "recall": float(recall_score(labels, preds, average="macro", zero_division=0)),
        "roc_auc": float(roc_auc_score(labels, probs[:, 1])),
    }

In [ ]:
# === 7. TrainingArguments + Trainer ===
from transformers import TrainingArguments, Trainer

OUTPUT_DIR = Path("models/checkpoints/xlmr-lusaber")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    report_to="none",
    logging_steps=25,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# === 8. Train ===
# WAIT: confirm with the project lead before running this cell. The
# fine-tune takes ~25–40 min on a T4 for ~2 300 rows × 5 epochs.
train_result = trainer.train()
log.info("training done: %s", train_result.metrics)

In [ ]:
# === 9. Hold-out eval + save ===
test_metrics = trainer.evaluate(ds_test, metric_key_prefix="test")
log.info("held-out test: %s", json.dumps(test_metrics, indent=2))

SAVE_DIR = Path("models/checkpoints/xlmr-lusaber-best")
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
log.info("saved best model -> %s", SAVE_DIR)

## Phase 3b — TF-IDF + Phase-2 features → LightGBM (5-fold CV)

TF-IDF on char-word boundaries, concatenated with the Lusaber Phase-2
feature vector (`models.features.FeatureExtractor`). Cross-validated
with `StratifiedKFold(n_splits=5)`.

In [ ]:
# === 10. Phase-2 feature extraction over all rows ===
from tqdm import tqdm
from models.features import FeatureExtractor, FEATURE_ORDER

fx = FeatureExtractor()

def _phase2(part_df):
    rows = []
    for _, r in tqdm(part_df.iterrows(), total=len(part_df), desc="phase2"):
        fv = fx.extract(title=r["title"], body_text=r["body_text"], url=(r["url"] or None))
        rows.append(fv.as_list(order=FEATURE_ORDER))
    return np.asarray(rows, dtype=np.float32)

X2_train = _phase2(train_df)
X2_val   = _phase2(val_df)
X2_test  = _phase2(test_df)
log.info("phase2 shapes: train=%s val=%s test=%s", X2_train.shape, X2_val.shape, X2_test.shape)

In [ ]:
# === 11. TF-IDF + concat + LightGBM 5-fold CV ===
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

def _docs(part_df):
    return [format_input(t, b) for t, b in zip(part_df["title"], part_df["body_text"])]

tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), analyzer="char_wb")
Xtf_train = tfidf.fit_transform(_docs(train_df))
Xtf_val   = tfidf.transform(_docs(val_df))
Xtf_test  = tfidf.transform(_docs(test_df))

X_train = hstack([Xtf_train, csr_matrix(X2_train)]).tocsr()
X_val   = hstack([Xtf_val,   csr_matrix(X2_val)]).tocsr()
X_test  = hstack([Xtf_test,  csr_matrix(X2_test)]).tocsr()
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values
log.info("feature matrix train shape: %s", X_train.shape)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_f1 = []
for fold, (tr, va) in enumerate(skf.split(X_train, y_train), 1):
    clf = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=-1, random_state=SEED)
    clf.fit(X_train[tr], y_train[tr], eval_set=[(X_train[va], y_train[va])])
    p = clf.predict(X_train[va])
    f = f1_score(y_train[va], p, average="macro")
    cv_f1.append(f)
    log.info("fold %d  macro-F1=%.4f", fold, f)
log.info("5-fold CV macro-F1: mean=%.4f std=%.4f", float(np.mean(cv_f1)), float(np.std(cv_f1)))

In [ ]:
# === 12. Final LightGBM + calibration ===
from sklearn.calibration import CalibratedClassifierCV

base_lgbm = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, random_state=SEED)
lgbm_cal = CalibratedClassifierCV(base_lgbm, method="sigmoid", cv=5)
lgbm_cal.fit(X_train, y_train)
log.info("calibrated LightGBM fitted")

lgbm_probs_val  = lgbm_cal.predict_proba(X_val)[:, 1]
lgbm_probs_test = lgbm_cal.predict_proba(X_test)[:, 1]

## Phase 3c — Soft-vote ensemble + score mapping

```
final_prob = 0.65 × transformer_prob + 0.35 × lgbm_prob
credibility_score = round(100 × final_prob, 2)
verdict =  <40 → LIKELY DISINFORMATION
          40–65 → UNCERTAIN
          >65   → LIKELY CREDIBLE
```

In [ ]:
# === 13. Transformer probs on val/test ===
import torch.nn.functional as F

def transformer_probs(hf_dataset) -> np.ndarray:
    pred = trainer.predict(hf_dataset)
    probs = F.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    return probs[:, 1]

xlmr_probs_val  = transformer_probs(ds_val)
xlmr_probs_test = transformer_probs(ds_test)

In [ ]:
# === 14. Soft-vote + score mapping + held-out metrics ===
from sklearn.metrics import classification_report, roc_auc_score

def soft_vote(p_xlmr, p_lgbm, w_xlmr=0.65, w_lgbm=0.35):
    return w_xlmr * p_xlmr + w_lgbm * p_lgbm

def to_score_and_verdict(prob: np.ndarray):
    score = np.round(100.0 * prob, 2)
    verdict = np.where(score < 40, "LIKELY DISINFORMATION",
                np.where(score > 65, "LIKELY CREDIBLE", "UNCERTAIN"))
    return score, verdict

p_test = soft_vote(xlmr_probs_test, lgbm_probs_test)
pred_test = (p_test >= 0.5).astype(int)
scores, verdicts = to_score_and_verdict(p_test)

print(classification_report(y_test, pred_test, target_names=["disinfo", "credible"], digits=4))
print("ROC-AUC :", roc_auc_score(y_test, p_test))
print("\nverdict distribution on held-out test:")
print(pd.Series(verdicts).value_counts())

In [ ]:
# === 15. Persist ensemble artifacts ===
import joblib
ARTIFACTS = Path("models/checkpoints")
joblib.dump(tfidf,     ARTIFACTS / "tfidf.joblib")
joblib.dump(lgbm_cal,  ARTIFACTS / "lgbm_calibrated.joblib")
(ARTIFACTS / "ensemble.json").write_text(json.dumps({
    "weights": {"transformer": 0.65, "lgbm": 0.35},
    "thresholds": {"disinfo_below": 40, "credible_above": 65},
    "base_model": BASE_MODEL,
    "feature_order": list(FEATURE_ORDER),
    "cv_f1_mean": float(np.mean(cv_f1)),
    "test_metrics": {
        "f1_macro": float(f1_score(y_test, pred_test, average="macro")),
        "roc_auc": float(roc_auc_score(y_test, p_test)),
    },
}, indent=2))
log.info("artifacts saved to %s", ARTIFACTS)